In [ ]:
import cv2
import numpy as np

# ✅ URL de tu cámara IP
url = "http://192.168.135.186:8080/video"
# url = "http://192.168.0.10:8080/video"  # cámara IP Android

cap = cv2.VideoCapture(url)

if not cap.isOpened():
    print("❌ No se pudo abrir la cámara IP")
    exit()

# ✅ Configurar ArUco (API NUEVA que tu OpenCV sí tiene)
aruco = cv2.aruco
aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)
params = aruco.DetectorParameters()
detector = cv2.aruco.ArucoDetector(aruco_dict, params)

# ✅ Matriz de cámara ficticia (hasta calibrar)
K = np.array([[800, 0, 320],
              [0, 800, 240],
              [0,   0,   1]], dtype=float)
dist = np.zeros(5)

# ✅ Tamaño real de tu marcador
MARKER_SIZE = 2.7  # cm

def solve_pose(corners):
    """Estimación de pose para un marcador usando solvePnP (siempre disponible)"""
    half = MARKER_SIZE / 2.0
    obj_points = np.array([
        [-half,  half, 0],
        [ half,  half, 0],
        [ half, -half, 0],
        [-half, -half, 0]
    ], dtype=np.float32)

    img_points = corners.reshape((4,2)).astype(np.float32)

    ok, rvec, tvec = cv2.solvePnP(
        obj_points, img_points, K, dist,
        flags=cv2.SOLVEPNP_IPPE_SQUARE
    )
    return rvec, tvec.reshape(3,1)

print("✅ Sistema funcionando en tiempo real")

while True:
    ret, frame = cap.read()
    if not ret:
        print("⚠ No se pudo leer frame")
        break
    frame = cv2.resize(frame, (640, 360)) 
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # ✅ Detectar ArUco
    corners, ids, _ = detector.detectMarkers(gray)

    if ids is not None:
        ids = ids.flatten()
        cv2.aruco.drawDetectedMarkers(frame, corners, ids.reshape(-1,1))

        # ✅ Obtener poses de todos los marcadores
        poses = {}
        centers = {}

        for i, marker_id in enumerate(ids):
            rvec, tvec = solve_pose(corners[i][0])
            poses[marker_id] = (rvec, tvec)

            # centro para dibujar en imagen
            cx = int(np.mean(corners[i][0][:,0]))
            cy = int(np.mean(corners[i][0][:,1]))
            centers[marker_id] = (cx, cy)

        # ✅ Seleccionar marcador origen
        ORIGIN = 0

        if ORIGIN in poses:
            r0, t0 = poses[ORIGIN]
            R0, _ = cv2.Rodrigues(r0)
            t0 = t0.reshape(3,1)

            puntos_2D = {}

            # ✅ Convertir todos a coordenadas relativas en 2D (X,Y cm)
            for marker_id, (rvec, tvec) in poses.items():
                t = tvec.reshape(3,1)
                relative = R0.T @ (t - t0)

                x = float(relative[0])
                y = float(relative[1])
                puntos_2D[marker_id] = (x, y)

            # ✅ Si hay 4 marcadores: dibujar el área
            if len(puntos_2D) >= 3:

                pts = np.array(list(puntos_2D.values()))
                centro = np.mean(pts, axis=0)
                ang = np.arctan2(pts[:,1]-centro[1], pts[:,0]-centro[0])
                orden = np.argsort(ang)
                ids_ordenados = np.array(list(puntos_2D.keys()))[orden]

                # ✅ Dibujar polígono conectando los pixeles
                poly_pts = [centers[i] for i in ids_ordenados]
                for i in range(len(poly_pts)):
                    p1 = poly_pts[i]
                    p2 = poly_pts[(i+1)%len(poly_pts)]
                    cv2.line(frame, p1, p2, (255,0,0), 3)

            # ✅ Mostrar coordenadas en consola
            print("\n📌 Coordenadas 2D reales (cm):")
            for k,(x,y) in puntos_2D.items():
                print(f"ID {k}: X={x:.1f}cm  Y={y:.1f}cm")

    cv2.imshow("ArUco + Área (Tiempo Real)", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


✅ Sistema funcionando en tiempo real

📌 Coordenadas 2D reales (cm):
ID 2: X=0.5cm  Y=14.2cm
ID 3: X=11.2cm  Y=14.3cm
ID 0: X=0.0cm  Y=0.0cm


C:\Users\CAROL\AppData\Local\Temp\ipykernel_33152\3211679724.py:92: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  x = float(relative[0])
C:\Users\CAROL\AppData\Local\Temp\ipykernel_33152\3211679724.py:93: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  y = float(relative[1])



📌 Coordenadas 2D reales (cm):
ID 2: X=0.5cm  Y=14.2cm
ID 3: X=11.2cm  Y=14.3cm
ID 0: X=0.0cm  Y=0.0cm

📌 Coordenadas 2D reales (cm):
ID 2: X=0.5cm  Y=14.2cm
ID 0: X=0.0cm  Y=0.0cm

📌 Coordenadas 2D reales (cm):
ID 2: X=0.5cm  Y=14.2cm
ID 3: X=11.2cm  Y=14.3cm
ID 0: X=0.0cm  Y=0.0cm

📌 Coordenadas 2D reales (cm):
ID 2: X=0.5cm  Y=14.2cm
ID 0: X=0.0cm  Y=0.0cm

📌 Coordenadas 2D reales (cm):
ID 2: X=0.5cm  Y=14.2cm
ID 3: X=11.1cm  Y=14.2cm
ID 0: X=0.0cm  Y=0.0cm

📌 Coordenadas 2D reales (cm):
ID 2: X=0.5cm  Y=14.2cm
ID 3: X=11.1cm  Y=14.2cm
ID 0: X=0.0cm  Y=0.0cm

📌 Coordenadas 2D reales (cm):
ID 2: X=0.5cm  Y=14.2cm
ID 0: X=0.0cm  Y=0.0cm

📌 Coordenadas 2D reales (cm):
ID 2: X=0.5cm  Y=14.2cm
ID 3: X=11.2cm  Y=14.3cm
ID 0: X=0.0cm  Y=0.0cm

📌 Coordenadas 2D reales (cm):
ID 2: X=0.5cm  Y=14.2cm
ID 3: X=11.2cm  Y=14.3cm
ID 0: X=0.0cm  Y=0.0cm

📌 Coordenadas 2D reales (cm):
ID 2: X=0.5cm  Y=14.2cm
ID 3: X=11.2cm  Y=14.3cm
ID 0: X=0.0cm  Y=0.0cm

📌 Coordenadas 2D reales (cm):
ID 2: X=0.5cm 